# Information Retrieval — Phase 2
**Team:** Yanis Hemdane, Rayane Khatim, Nour el imene Khelassi, Aya Chihoub

## Pipeline

1. **Classifier** — TF-IDF + Logistic Regression → predicts query category (~99.7%).
2. **First-stage retrieval** — 8-way weighted RRF with two embedding models:
   - BM25 on short text | BM25 on content | BM25+PRF
   - MiniLM-L12 embeddings (short + content) | mpnet-v2 embeddings (short + content)
   - Global mpnet embeddings
3. **Second-stage re-ranking** — LightGBM LambdaRank with 5-fold CV.
   10 features: BM25 (3), MiniLM embeddings (2), mpnet embeddings (2), TF-IDF sim, term overlap, RRF rank.
4. **Auto-tune k** on out-of-fold predictions → writes `solutions_Hemdane_Khatim_Khelassi_Chihoub.csv`.

---

### Phase 2 — Short answers (course brief)

| # | Question | Answer |
|---|----------|--------|
| 1 | **What model for classification?** | TF-IDF (1-2 grams) + **Logistic Regression** (`class_weight='balanced'`, `C=5.0`). |
| 2 | **Training data?** | All **documents** + all **training queries** (merged `title`, `text`, `tags`). |
| 3 | **Features?** | TF-IDF word-level unigrams and bigrams on the merged `content` string. |
| 4 | **Accuracy?** | Printed below (~99.7 %). |
| — | **How does the classifier help?** | We **scope** BM25/embedding indices to the predicted category. |
| — | **Title matching?** | Query `text` is matched against doc `title` — both are short descriptive strings. |
| — | **Embedding models?** | `all-MiniLM-L12-v2` (fast) + `all-mpnet-base-v2` (high quality). |
| — | **Learning to rank?** | LightGBM LambdaRank with 5-fold CV, 10 features, heavily regularized. |

---
## Cell 1 — Install

In [1]:
!pip install -q rank_bm25 sentence-transformers tqdm lightgbm

---
## Cell 2 — Imports

In [2]:
import csv, json, os, re, time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from rank_bm25 import BM25Plus
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import normalize
from tqdm import tqdm

try:
    import lightgbm as lgb
    USE_LGB = True
    print(f'LightGBM {lgb.__version__}')
except ImportError:
    USE_LGB = False
    from sklearn.ensemble import GradientBoostingClassifier
    print('LightGBM not found, using sklearn fallback')

np.random.seed(42)
print('All imports OK')

LightGBM 4.6.0
All imports OK


---
## Cell 3 — Configuration

In [3]:
POOL       = 500
GLOBAL_TOP = 100
PRF_DOCS   = 7
PRF_TERMS  = 30
LTR_POOL   = 300
TUNE_K     = True
K          = 10

EMB_MODEL_1 = 'all-MiniLM-L12-v2'
EMB_MODEL_2 = 'all-mpnet-base-v2'

_kaggle = Path('/kaggle/working').exists()
CACHE_DIR   = Path('/kaggle/working/cache') if _kaggle else Path('cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = Path('/kaggle/working/solutions_Hemdane_Khatim_Khelassi_Chihoub.csv') if _kaggle else Path('solutions_Hemdane_Khatim_Khelassi_Chihoub.csv')

print(f'POOL={POOL}  GLOBAL_TOP={GLOBAL_TOP}  LTR_POOL={LTR_POOL}')
print(f'PRF_DOCS={PRF_DOCS}  PRF_TERMS={PRF_TERMS}')
print(f'Bi-encoder 1: {EMB_MODEL_1}')
print(f'Bi-encoder 2: {EMB_MODEL_2}')

POOL=500  GLOBAL_TOP=100  LTR_POOL=300
PRF_DOCS=7  PRF_TERMS=30
Bi-encoder 1: all-MiniLM-L12-v2
Bi-encoder 2: all-mpnet-base-v2


---
## Cell 4 — Load data

In [4]:
def find_data_dir():
    for c in [
        Path('/kaggle/input/competitions/retrieval-engine-competition'),
        Path('/kaggle/input/retrieval-engine-competition'),
        Path('/kaggle/input'),
        Path('data/retrieval-engine-competition'),
        Path('data'), Path('.'),
    ]:
        if (c / 'docs.json').is_file():
            return c
    for root, _, files in os.walk('/kaggle/input'):
        if 'docs.json' in files:
            return Path(root)
    raise FileNotFoundError('docs.json not found.')

DATA_DIR = find_data_dir()
print(f'Data: {DATA_DIR}')

df_docs          = pd.read_json(DATA_DIR / 'docs.json')
df_queries_train = pd.read_json(DATA_DIR / 'queries_train.json')
df_queries_test  = pd.read_json(DATA_DIR / 'queries_test.json')

with open(DATA_DIR / 'qgts_train.json') as f:
    raw_gt = json.load(f)
ground_truth = {
    qid: [item['doc_id'] for item in data['relevant_doc_ids']]
    for qid, data in raw_gt.items()
}

rel_counts = [len(v) for v in ground_truth.values()]
print(f'Docs={len(df_docs):,}  TrainQ={len(df_queries_train)}  TestQ={len(df_queries_test)}')
print(f'Relevant/query: avg={np.mean(rel_counts):.1f}  min={min(rel_counts)}  max={max(rel_counts)}')
print(f'Categories: {sorted(df_docs["category"].unique())}')

Data: /kaggle/input/competitions/retrieval-engine-competition
Docs=216,041  TrainQ=327  TestQ=141
Relevant/query: avg=9.1  min=4  max=262
Categories: ['android', 'gaming', 'programmers', 'tex', 'unix']


---
## Cell 5 — Preprocessing

Query `title` is always empty — the actual query is `text`.
Short-text matching: `query.text` vs `doc.title`.

In [5]:
def merge_fields(row):
    title    = str(row.get('title', '') or '')
    text     = str(row.get('text',  '') or '')
    tags     = row.get('tags', [])
    tags_str = ' '.join(tags) if isinstance(tags, list) else str(tags or '')
    return ' '.join(f'{title} {text} {tags_str}'.split())

def get_short_text(row, is_query=False):
    if is_query:
        return str(row.get('text', '') or '').strip()
    title = str(row.get('title', '') or '')
    tags  = row.get('tags', [])
    tags_str = ' '.join(tags) if isinstance(tags, list) else str(tags or '')
    return ' '.join(f'{title} {tags_str}'.split())

def clean_text(text):
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    return ' '.join(text.split())

for df, is_q in [(df_docs, False), (df_queries_train, True), (df_queries_test, True)]:
    df['content']       = df.apply(merge_fields, axis=1)
    df['content_clean'] = df['content'].apply(clean_text)
    df['short_text']    = df.apply(lambda r, q=is_q: get_short_text(r, is_query=q), axis=1)
    df['short_clean']   = df['short_text'].apply(clean_text)

doc_ids         = df_docs['id'].tolist()
query_ids_train = df_queries_train['id'].tolist()
query_ids_test  = df_queries_test['id'].tolist()
id_to_cat       = dict(zip(df_docs['id'], df_docs['category']))
doc_id_to_idx   = {did: i for i, did in enumerate(doc_ids)}

print('Preprocessing done.')
print(f'Doc   short: {df_docs["short_text"].iloc[0][:80]}')
print(f'Query short: {df_queries_train["short_text"].iloc[0][:80]}')

Preprocessing done.
Doc   short: MikTex Download Failure - toptesi.tar.lzma miktex
Query short: Want to try reformatting Damaged SD Card


---
## Cell 6 — Classifier

In [6]:
all_contents = df_docs['content'].tolist() + df_queries_train['content'].tolist()
all_labels   = df_docs['category'].tolist() + df_queries_train['category'].tolist()

print(f'Training classifier on {len(all_contents):,} samples...')
clf_vec = TfidfVectorizer(max_features=15000, sublinear_tf=True, ngram_range=(1, 2))
X_all   = clf_vec.fit_transform(all_contents)
clf     = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced', C=5.0)
clf.fit(X_all, all_labels)

pred_cats_train = clf.predict(clf_vec.transform(df_queries_train['content'].tolist())).tolist()
pred_cats_test  = clf.predict(clf_vec.transform(df_queries_test['content'].tolist())).tolist()
true_cats_train = df_queries_train['category'].tolist()

clf_acc = float((np.array(pred_cats_train) == np.array(true_cats_train)).mean())
print(f'Classifier accuracy: {clf_acc:.4f}')
print(classification_report(true_cats_train, pred_cats_train))

# TF-IDF matrix for LTR feature (separate from classifier)
print('Building TF-IDF retrieval matrix...')
tfidf_ret = TfidfVectorizer(max_features=50000, sublinear_tf=True, ngram_range=(1, 2))
D_tfidf = normalize(tfidf_ret.fit_transform(df_docs['content_clean']))
Q_train_tfidf = normalize(tfidf_ret.transform(df_queries_train['content_clean']))
Q_test_tfidf  = normalize(tfidf_ret.transform(df_queries_test['content_clean']))
print(f'TF-IDF matrix: {D_tfidf.shape}')

Training classifier on 216,368 samples...
Classifier accuracy: 0.9969
              precision    recall  f1-score   support

     android       1.00      0.98      0.99        57
      gaming       1.00      1.00      1.00        41
 programmers       0.98      1.00      0.99        52
         tex       1.00      1.00      1.00       130
        unix       1.00      1.00      1.00        47

    accuracy                           1.00       327
   macro avg       1.00      1.00      1.00       327
weighted avg       1.00      1.00      1.00       327

Building TF-IDF retrieval matrix...
TF-IDF matrix: (216041, 50000)


---
## Cell 7 — Per-category BM25+ indices

In [7]:
cat_to_indices = defaultdict(list)
for i, cat in enumerate(df_docs['category']):
    cat_to_indices[cat].append(i)

categories = sorted(cat_to_indices.keys())
for cat in categories:
    print(f'  {cat}: {len(cat_to_indices[cat]):,} docs')

cat_bm25_content   = {}
cat_bm25_short     = {}
cat_doc_ids        = {}
cat_content_tokens = {}

for cat in tqdm(categories, desc='BM25 indices'):
    indices     = cat_to_indices[cat]
    content_tok = [df_docs['content_clean'].iloc[i].split() for i in indices]
    short_tok   = [df_docs['short_clean'].iloc[i].split()   for i in indices]

    cat_bm25_content[cat] = BM25Plus(content_tok)
    cat_bm25_short[cat]   = BM25Plus(short_tok)
    cat_doc_ids[cat]      = [doc_ids[i] for i in indices]
    cat_content_tokens[cat] = content_tok

print('BM25+ indices ready.')

  android: 22,998 docs
  gaming: 45,301 docs
  programmers: 32,176 docs
  tex: 68,184 docs
  unix: 47,382 docs


BM25 indices: 100%|██████████| 5/5 [00:15<00:00,  3.17s/it]

BM25+ indices ready.


---
## Cell 8 — Bi-encoder embeddings (two models)

Encode with both `all-MiniLM-L12-v2` (fast, proven) and `all-mpnet-base-v2` (highest quality).
Both sets of embeddings are used as RRF channels and LTR features.

In [8]:
def encode_model(model_name):
    print(f'\n=== {model_name} ===')
    encoder = SentenceTransformer(model_name)
    dim = encoder.get_sentence_embedding_dimension()
    print(f'Dim: {dim}')
    mtag = model_name.replace('/', '_').replace('-', '_')

    def enc(texts, path, label):
        if path.is_file():
            print(f'  Cache: {path.name}')
            return np.load(str(path))
        print(f'  Encoding {label} ({len(texts):,})...')
        embs = encoder.encode(texts, show_progress_bar=True, batch_size=64,
                              normalize_embeddings=True)
        np.save(str(path), embs)
        return embs

    dc = enc(df_docs['content'].tolist(),        CACHE_DIR / f'doc_content_{mtag}.npy', 'doc content')
    ds = enc(df_docs['short_text'].tolist(),      CACHE_DIR / f'doc_short_{mtag}.npy',   'doc short')
    trc = enc(df_queries_train['content'].tolist(), CACHE_DIR / f'trq_content_{mtag}.npy', 'train-q content')
    trs = enc(df_queries_train['short_text'].tolist(), CACHE_DIR / f'trq_short_{mtag}.npy', 'train-q short')
    tec = enc(df_queries_test['content'].tolist(),  CACHE_DIR / f'teq_content_{mtag}.npy', 'test-q content')
    tes = enc(df_queries_test['short_text'].tolist(), CACHE_DIR / f'teq_short_{mtag}.npy', 'test-q short')

    cat_ce = {cat: dc[cat_to_indices[cat]] for cat in categories}
    cat_se = {cat: ds[cat_to_indices[cat]] for cat in categories}

    del encoder
    return dc, ds, trc, trs, tec, tes, cat_ce, cat_se

(all_doc_content_embs_1, all_doc_short_embs_1,
 trq_content_embs_1, trq_short_embs_1,
 teq_content_embs_1, teq_short_embs_1,
 cat_content_embs_1, cat_short_embs_1) = encode_model(EMB_MODEL_1)

(all_doc_content_embs_2, all_doc_short_embs_2,
 trq_content_embs_2, trq_short_embs_2,
 teq_content_embs_2, teq_short_embs_2,
 cat_content_embs_2, cat_short_embs_2) = encode_model(EMB_MODEL_2)

print('\nAll embeddings ready (two models).')


=== all-MiniLM-L12-v2 ===


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

  Encoding test-q content (141)...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

  Encoding test-q short (141)...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]


All embeddings ready (two models).


---
## Cell 9 — First-stage retrieval (8-way RRF)

Fuses BM25 (3 channels) + MiniLM embeddings (2) + mpnet embeddings (2) + global mpnet (1).

In [9]:
def weighted_rrf(ranked_lists, weights, rrf_k=60):
    scores = {}
    for ranked, w in zip(ranked_lists, weights):
        for rank, did in enumerate(ranked, 1):
            scores[did] = scores.get(did, 0.0) + w / (rrf_k + rank)
    return sorted(scores, key=lambda x: scores[x], reverse=True)

def expand_query_prf(q_tokens, bm25_idx, doc_tok_list, n_docs=5, n_terms=25):
    scores = bm25_idx.get_scores(q_tokens)
    top_idx = np.argsort(scores)[-n_docs:][::-1]
    q_set = set(q_tokens)
    tc = Counter()
    for idx in top_idx:
        for t in doc_tok_list[idx]:
            if t not in q_set and len(t) > 2:
                tc[t] += 1
    return q_tokens + [t for t, _ in tc.most_common(n_terms)]

def _topk_ids(scores_arr, cat, pool):
    top = np.argsort(scores_arr)[-pool:][::-1]
    return [cat_doc_ids[cat][j] for j in top]

def retrieve(q_cc, q_sc, q_ce_1, q_se_1, q_ce_2, q_se_2, pred_cat, pool, global_top):
    cat = pred_cat
    q_ct = q_cc.split()
    q_st = q_sc.split()
    # BM25 channels
    l1 = _topk_ids(cat_bm25_short[cat].get_scores(q_st), cat, pool)
    l2 = _topk_ids(cat_bm25_content[cat].get_scores(q_ct), cat, pool)
    exp = expand_query_prf(q_ct, cat_bm25_content[cat], cat_content_tokens[cat], PRF_DOCS, PRF_TERMS)
    l3 = _topk_ids(cat_bm25_content[cat].get_scores(exp), cat, pool)
    # MiniLM channels
    l4 = _topk_ids(q_se_1 @ cat_short_embs_1[cat].T, cat, pool)
    l5 = _topk_ids(q_ce_1 @ cat_content_embs_1[cat].T, cat, pool)
    # mpnet channels
    l6 = _topk_ids(q_se_2 @ cat_short_embs_2[cat].T, cat, pool)
    l7 = _topk_ids(q_ce_2 @ cat_content_embs_2[cat].T, cat, pool)
    # Global (mpnet)
    sg = q_ce_2 @ all_doc_content_embs_2.T
    tg = np.argsort(sg)[-global_top:][::-1]
    l8 = [doc_ids[j] for j in tg]
    return weighted_rrf(
        [l1, l2, l3, l4, l5, l6, l7, l8],
        [2.0, 1.0, 0.8, 1.5, 1.0, 2.0, 1.5, 0.5])

print('Retrieving train queries...')
t0 = time.time()
fused_train = [
    retrieve(df_queries_train['content_clean'].iloc[i],
             df_queries_train['short_clean'].iloc[i],
             trq_content_embs_1[i], trq_short_embs_1[i],
             trq_content_embs_2[i], trq_short_embs_2[i],
             pred_cats_train[i], POOL, GLOBAL_TOP)
    for i in tqdm(range(len(query_ids_train)))]
print(f'Train: {time.time()-t0:.1f}s')

print('Retrieving test queries...')
t0 = time.time()
fused_test = [
    retrieve(df_queries_test['content_clean'].iloc[i],
             df_queries_test['short_clean'].iloc[i],
             teq_content_embs_1[i], teq_short_embs_1[i],
             teq_content_embs_2[i], teq_short_embs_2[i],
             pred_cats_test[i], POOL, GLOBAL_TOP)
    for i in tqdm(range(len(query_ids_test)))]
print(f'Test: {time.time()-t0:.1f}s')

Retrieving train queries...


100%|██████████| 327/327 [07:34<00:00,  1.39s/it]


Train: 454.3s
Retrieving test queries...


100%|██████████| 141/141 [03:26<00:00,  1.46s/it]

Test: 206.4s


---
## Cell 10 — Learning-to-Rank (5-fold CV, 10 features)

Features from both embedding models, BM25, TF-IDF cosine similarity, and term overlap.
5-fold CV on queries for honest estimation. Final model trained on all queries.

In [10]:
def compute_metrics(topk_lists, query_ids):
    R, P, M, A = [], [], [], []
    for i, qid in enumerate(query_ids):
        rel     = set(ground_truth.get(qid, []))
        ret     = topk_lists[i]
        hit     = set(ret) & rel
        R.append(len(hit) / len(rel) if rel else 0.0)
        P.append(len(hit) / len(ret) if ret else 0.0)
        A.append(1.0 if hit else 0.0)
        mrr = 0.0
        for rank, d in enumerate(ret, 1):
            if d in rel:
                mrr = 1.0 / rank
                break
        M.append(mrr)
    return {'Recall': float(np.mean(R)), 'Precision': float(np.mean(P)),
            'MRR': float(np.mean(M)), 'Accuracy': float(np.mean(A))}

FEAT_NAMES = ['bm25_content', 'bm25_short', 'bm25_prf',
              'emb1_content', 'emb1_short',
              'emb2_content', 'emb2_short',
              'tfidf_sim', 'term_overlap', 'rrf_rank']

def extract_features(q_cc, q_sc, q_ce_1, q_se_1, q_ce_2, q_se_2,
                     q_tfidf, pred_cat, candidates):
    cat = pred_cat
    q_ct = q_cc.split()
    q_st = q_sc.split()
    q_set = set(q_ct)

    bm25c = cat_bm25_content[cat].get_scores(q_ct)
    bm25s = cat_bm25_short[cat].get_scores(q_st)
    exp   = expand_query_prf(q_ct, cat_bm25_content[cat],
                             cat_content_tokens[cat], PRF_DOCS, PRF_TERMS)
    bm25p = cat_bm25_content[cat].get_scores(exp)

    embc_1 = q_ce_1 @ cat_content_embs_1[cat].T
    embs_1 = q_se_1 @ cat_short_embs_1[cat].T
    embc_2 = q_ce_2 @ cat_content_embs_2[cat].T
    embs_2 = q_se_2 @ cat_short_embs_2[cat].T

    cat_ids = cat_doc_ids[cat]
    id2loc  = {did: j for j, did in enumerate(cat_ids)}

    cat_indices_arr = cat_to_indices[cat]
    D_cat_tfidf = D_tfidf[cat_indices_arr]
    tfidf_sims = (q_tfidf @ D_cat_tfidf.T).toarray().ravel()

    feats, valid = [], []
    for rank, did in enumerate(candidates):
        if did in id2loc:
            j = id2loc[did]
            toks = cat_content_tokens[cat][j]
            overlap = len(q_set & set(toks)) / max(len(q_set), 1)
            feats.append([
                bm25c[j], bm25s[j], bm25p[j],
                float(embc_1[j]), float(embs_1[j]),
                float(embc_2[j]), float(embs_2[j]),
                float(tfidf_sims[j]),
                overlap, rank / LTR_POOL
            ])
            valid.append(did)
        else:
            gi = doc_id_to_idx.get(did, -1)
            if gi >= 0:
                ts = float((q_tfidf @ D_tfidf[gi].T).toarray()[0, 0])
                feats.append([
                    0.0, 0.0, 0.0,
                    float(q_ce_1 @ all_doc_content_embs_1[gi]),
                    float(q_se_1 @ all_doc_short_embs_1[gi]),
                    float(q_ce_2 @ all_doc_content_embs_2[gi]),
                    float(q_se_2 @ all_doc_short_embs_2[gi]),
                    ts, 0.0, rank / LTR_POOL
                ])
                valid.append(did)
    return valid, np.array(feats) if feats else np.empty((0, len(FEAT_NAMES)))

# ── Build per-query feature matrices ──
print('Extracting train features...')
t0 = time.time()
per_query_vids = []
per_query_feats = []
per_query_labels = []

for i in tqdm(range(len(query_ids_train))):
    qid = query_ids_train[i]
    rel = set(ground_truth.get(qid, []))
    cands = fused_train[i][:LTR_POOL]
    vids, fmat = extract_features(
        df_queries_train['content_clean'].iloc[i],
        df_queries_train['short_clean'].iloc[i],
        trq_content_embs_1[i], trq_short_embs_1[i],
        trq_content_embs_2[i], trq_short_embs_2[i],
        Q_train_tfidf[i],
        pred_cats_train[i], cands)
    labels = np.array([1 if d in rel else 0 for d in vids])
    per_query_vids.append(vids)
    per_query_feats.append(fmat)
    per_query_labels.append(labels)

total_samples = sum(len(v) for v in per_query_vids)
pos_rate = sum(l.sum() for l in per_query_labels) / total_samples
print(f'Features: {time.time()-t0:.1f}s  |  {total_samples} samples  |  pos rate: {pos_rate:.3f}')

# ── LightGBM hyperparameters ──
lgb_params = dict(
    objective='lambdarank', metric='ndcg',
    n_estimators=80, learning_rate=0.05,
    max_depth=4, num_leaves=12,
    min_child_samples=20, reg_alpha=0.3, reg_lambda=3.0,
    subsample=0.7, colsample_bytree=0.8, random_state=42,
    ndcg_eval_at=[1, 3, 5, 10],
    verbose=-1,
)

# ── 5-fold cross-validation ──
from sklearn.model_selection import KFold

n_queries = len(query_ids_train)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_rankings = [None] * n_queries

print(f'\n5-fold CV on {n_queries} queries...')
for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(range(n_queries))):
    X_fold = np.vstack([per_query_feats[i] for i in tr_idx])
    y_fold = np.concatenate([per_query_labels[i] for i in tr_idx])
    g_fold = [len(per_query_vids[i]) for i in tr_idx]

    if USE_LGB:
        fold_ranker = lgb.LGBMRanker(**lgb_params)
        fold_ranker.fit(X_fold, y_fold, group=g_fold)
    else:
        fold_ranker = GradientBoostingClassifier(
            n_estimators=80, max_depth=4, learning_rate=0.05,
            subsample=0.7, random_state=42)
        fold_ranker.fit(X_fold, y_fold)

    for i in val_idx:
        fmat = per_query_feats[i]
        vids = per_query_vids[i]
        if len(vids) > 0:
            if USE_LGB:
                scores = fold_ranker.predict(fmat)
            else:
                scores = fold_ranker.predict_proba(fmat)[:, 1]
            order = np.argsort(scores)[::-1]
            reranked = [vids[j] for j in order]
        else:
            reranked = []
        tail = [d for d in fused_train[i][LTR_POOL:] if d not in set(reranked)]
        oof_rankings[i] = reranked + tail

    val_trimmed = [oof_rankings[i][:10] for i in val_idx]
    val_qids = [query_ids_train[i] for i in val_idx]
    fm = compute_metrics(val_trimmed, val_qids)
    vest = 0.25 * (fm['Recall'] + fm['Precision'] + fm['MRR'] + clf_acc)
    print(f'  Fold {fold_idx+1}: MRR={fm["MRR"]:.4f}  Prec={fm["Precision"]:.4f}  '
          f'Recall={fm["Recall"]:.4f}  Est@10={vest:.4f}')

oof_m10 = compute_metrics([r[:10] for r in oof_rankings], query_ids_train)
oof_est = 0.25 * (oof_m10['Recall'] + oof_m10['Precision'] + oof_m10['MRR'] + clf_acc)
print(f'\nOOF overall @k=10: MRR={oof_m10["MRR"]:.4f}  Est={oof_est:.4f}')

# ── Train final model on ALL queries ──
print('\nTraining final model on all queries...')
X_all = np.vstack(per_query_feats)
y_all = np.concatenate(per_query_labels)
g_all = [len(v) for v in per_query_vids]

if USE_LGB:
    ranker = lgb.LGBMRanker(**lgb_params)
    ranker.fit(X_all, y_all, group=g_all)
    print('\nFeature importance:')
    for nm, imp in sorted(zip(FEAT_NAMES, ranker.feature_importances_), key=lambda x: -x[1]):
        print(f'  {nm:16s} {imp}')
else:
    ranker = GradientBoostingClassifier(
        n_estimators=80, max_depth=4, learning_rate=0.05,
        subsample=0.7, random_state=42)
    ranker.fit(X_all, y_all)
    print('sklearn GBC trained.')

# ── Re-rank test queries with final model ──
print('\nRe-ranking test queries...')
ltr_test = []
for i in tqdm(range(len(query_ids_test))):
    cands = fused_test[i][:LTR_POOL]
    vids, fmat = extract_features(
        df_queries_test['content_clean'].iloc[i],
        df_queries_test['short_clean'].iloc[i],
        teq_content_embs_1[i], teq_short_embs_1[i],
        teq_content_embs_2[i], teq_short_embs_2[i],
        Q_test_tfidf[i],
        pred_cats_test[i], cands)
    if len(vids) > 0:
        if USE_LGB:
            scores = ranker.predict(fmat)
        else:
            scores = ranker.predict_proba(fmat)[:, 1]
        order = np.argsort(scores)[::-1]
        reranked = [vids[j] for j in order]
    else:
        reranked = []
    tail = [d for d in fused_test[i][LTR_POOL:] if d not in set(reranked)]
    ltr_test.append(reranked + tail)

ltr_train = oof_rankings
print('LTR re-ranking done (test uses full model, train uses OOF).')

Extracting train features...


100%|██████████| 327/327 [08:08<00:00,  1.49s/it]
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


Features: 488.3s  |  98100 samples  |  pos rate: 0.018

5-fold CV on 327 queries...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitte

  Fold 1: MRR=0.4616  Prec=0.1970  Recall=0.2919  Est@10=0.4869


...

  Fold 4: MRR=0.4972  Prec=0.1877  Recall=0.3411  Est@10=0.5057


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitte

  Fold 5: MRR=0.4486  Prec=0.1923  Recall=0.3158  Est@10=0.4884

OOF overall @k=10: MRR=0.4736  Est=0.4973

Training final model on all queries...

Feature importance:
  emb2_short       179
  rrf_rank         101
  term_overlap     98
  tfidf_sim        97
  emb1_content     94
  emb2_content     79
  bm25_prf         73
  bm25_short       71
  emb1_short       59
  bm25_content     29

Re-ranking test queries...


  0%|          | 0/141 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
  1%|          | 1/141 [00:01<02:43,  1.17s/it]/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
  1%|▏         | 2/141 [00:02<03:02,  1.31s/it]/usr/local/lib/

LTR re-ranking done (test uses full model, train uses OOF).


---
## Cell 11 — Auto-tune k (on out-of-fold predictions)

In [11]:
if TUNE_K:
    print('Auto-tuning k on OOF LTR results...')
    print(f'{"k":>4}  {"Est":>8}  {"Recall":>8}  {"Prec":>8}  {"MRR":>8}  {"Acc":>6}')
    print('-' * 52)
    best_k, best_score = K, 0.0
    for k_try in [3, 4, 5, 7, 10, 12, 15, 20, 25, 30, 40, 50, 75, 100, 150]:
        trimmed = [ids[:k_try] for ids in ltr_train]
        m = compute_metrics(trimmed, query_ids_train)
        est = 0.25 * (m['Recall'] + m['Precision'] + m['MRR'] + clf_acc)
        print(f'{k_try:>4}  {est:>8.4f}  {m["Recall"]:>8.4f}  {m["Precision"]:>8.4f}  '
              f'{m["MRR"]:>8.4f}  {m["Accuracy"]:>6.3f}')
        if est > best_score:
            best_score, best_k = est, k_try
    K = best_k
    print(f'\n-> Best k = {K}  (estimated: {best_score:.4f})')

Auto-tuning k on OOF LTR results...
   k       Est    Recall      Prec       MRR     Acc
----------------------------------------------------
   3    0.4771    0.1636    0.3017    0.4460   0.563
   4    0.4827    0.1980    0.2798    0.4559   0.602
   5    0.4876    0.2303    0.2618    0.4614   0.630
   7    0.4936    0.2799    0.2311    0.4665   0.664
  10    0.4973    0.3252    0.1936    0.4736   0.725
  12    0.4998    0.3505    0.1758    0.4760   0.752
  15    0.5053    0.3877    0.1580    0.4784   0.786
  20    0.5093    0.4263    0.1338    0.4801   0.817
  25    0.5139    0.4609    0.1169    0.4808   0.832
  30    0.5151    0.4799    0.1027    0.4810   0.838
  40    0.5181    0.5099    0.0839    0.4816   0.859
  50    0.5229    0.5407    0.0717    0.4821   0.884
  75    0.5294    0.5849    0.0532    0.4824   0.899
 100    0.5354    0.6191    0.0428    0.4826   0.917
 150    0.5462    0.6732    0.0320    0.4827   0.936

-> Best k = 150  (estimated: 0.5462)


---
## Cell 12 — Final evaluation

In [12]:
final_train = [ids[:K] for ids in ltr_train]
m = compute_metrics(final_train, query_ids_train)
estimated = 0.25 * (m['Recall'] + m['Precision'] + m['MRR'] + clf_acc)

print('=' * 55)
print(f'FINAL EVALUATION  (k={K})')
print('=' * 55)
print(f'Recall              : {m["Recall"]:.4f}')
print(f'Precision           : {m["Precision"]:.4f}')
print(f'MRR                 : {m["MRR"]:.4f}')
print(f'Retrieval Accuracy  : {m["Accuracy"]:.4f}')
print(f'Classifier Accuracy : {clf_acc:.4f}')
print('-' * 55)
print(f'Estimated score     : {estimated:.4f}')
print('=' * 55)

FINAL EVALUATION  (k=150)
Recall              : 0.6732
Precision           : 0.0320
MRR                 : 0.4827
Retrieval Accuracy  : 0.9358
Classifier Accuracy : 0.9969
-------------------------------------------------------
Estimated score     : 0.5462


---
## Cell 13 — Comparison table (report)

In [13]:
# TF-IDF baseline (global)
tfidf_train_lists = []
for start in range(0, Q_train_tfidf.shape[0], 64):
    end = min(start + 64, Q_train_tfidf.shape[0])
    sims = (Q_train_tfidf[start:end] @ D_tfidf.T).toarray()
    for j in range(sims.shape[0]):
        top = np.argsort(sims[j])[-POOL:][::-1]
        tfidf_train_lists.append([doc_ids[idx] for idx in top])

# BM25 content only (scoped)
bm25_only = []
for i in range(len(query_ids_train)):
    cat = pred_cats_train[i]
    sc  = cat_bm25_content[cat].get_scores(df_queries_train['content_clean'].iloc[i].split())
    top = np.argsort(sc)[-POOL:][::-1]
    bm25_only.append([cat_doc_ids[cat][j] for j in top])

# Embeddings content only (mpnet, scoped)
emb_only = []
for i in range(len(query_ids_train)):
    cat  = pred_cats_train[i]
    sims = trq_content_embs_2[i] @ cat_content_embs_2[cat].T
    top  = np.argsort(sims)[-POOL:][::-1]
    emb_only.append([cat_doc_ids[cat][j] for j in top])

rrf_only = [ids[:K] for ids in fused_train]

comparisons = [
    ('TF-IDF (global)',                [ids[:K] for ids in tfidf_train_lists]),
    ('BM25+ content (scoped)',         [ids[:K] for ids in bm25_only]),
    ('mpnet embeddings (scoped)',      [ids[:K] for ids in emb_only]),
    ('8-way RRF (first stage)',        rrf_only),
    ('RRF + LightGBM LTR (OOF)',      final_train),
]

print(f'Comparison at k={K}\n')
print(f'{"Method":<38} {"Recall":>8} {"Prec":>8} {"MRR":>8} {"Est":>8}')
print('-' * 74)
for name, lists in comparisons:
    mm = compute_metrics(lists, query_ids_train)
    est = 0.25 * (mm['Recall'] + mm['Precision'] + mm['MRR'] + clf_acc)
    print(f'{name:<38} {mm["Recall"]:>8.4f} {mm["Precision"]:>8.4f} {mm["MRR"]:>8.4f} {est:>8.4f}')

Comparison at k=150

Method                                   Recall     Prec      MRR      Est
--------------------------------------------------------------------------
TF-IDF (global)                          0.2798   0.0123   0.1892   0.3696
BM25+ content (scoped)                   0.3245   0.0139   0.2334   0.3922
mpnet embeddings (scoped)                0.5104   0.0235   0.3160   0.4617
8-way RRF (first stage)                  0.6536   0.0307   0.4535   0.5337
RRF + LightGBM LTR (OOF)                 0.6732   0.0320   0.4827   0.5462


---
## Cell 14 — Write solutions_Hemdane_Khatim_Khelassi_Chihoub.csv

In [14]:
final_test = [ids[:K] for ids in ltr_test]

with open(OUTPUT_PATH, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['query_id', 'relevant_doc_ids', 'category'])
    for qid, doc_list, cat in zip(query_ids_test, final_test, pred_cats_test):
        writer.writerow([qid, json.dumps(doc_list), cat])

print(f'Written: {OUTPUT_PATH}')
print(f'Rows: {len(query_ids_test)}  |  k={K}  |  Size: {OUTPUT_PATH.stat().st_size:,} bytes')
print()
print('First 3 rows:')
with open(OUTPUT_PATH, 'r') as f:
    for i, line in enumerate(f):
        if i >= 4: break
        print(' ', line.strip()[:120])

Written: /kaggle/working/solutions_Hemdane_Khatim_Khelassi_Chihoub.csv
Rows: 141  |  k=150  |  Size: 1,029,671 bytes

First 3 rows:
  query_id,relevant_doc_ids,category
  4ffe16bc-5235-418d-9bf3-22d1f2c5796e_145437,"[""87a25467-3a02-4bc5-9d9d-af40d61b098e_159813"", ""c583f3cd-b1ca-4b74-ac3e
  1bb2bb20-7f45-4dcf-a94a-420c454f87b8_56473,"[""6b37c6e7-93d4-4e95-9da3-c9710e6d8191_91699"", ""71dabe33-86f4-4a8f-81e7-8
  6a9a342c-1275-4bb3-a818-8bcce53fac4f_34507,"[""43fa6e5a-6ad6-4701-ba44-68b513409ffc_17053"", ""b002e1f6-647d-4139-84d9-c
